In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")
from core.startup import init
engine, memory = init()

In [ ]:
with open("/workspace/Projects/cultivated-learning/core/interaction_loop.py", "r") as f:
    print(f.read())

In [ ]:
with open("/workspace/Projects/cultivated-learning/core/context_assembler.py", "r") as f:
    print(f.read())

In [ ]:
!cp /workspace/Projects/cultivated-learning/core/interaction_loop.py /workspace/Projects/cultivated-learning/core/interaction_loop_backup.py
print("Backup saved.")

In [ ]:
%%writefile /workspace/Projects/cultivated-learning/core/reflection.py
import time
from core.memory_store import MemoryUnit, MemoryType


class ReflectionEngine:
    """Post-interaction recursive self-analysis at increasing depths."""

    def __init__(self, engine, memory_store, max_depth=3):
        self.engine = engine
        self.memory = memory_store
        self.max_depth = max_depth
        self.directives = []
        self._load_directives()

    def _load_directives(self):
        """Load existing procedural directives from memory."""
        procedural = self.memory.retrieve_by_type(MemoryType.PROCEDURAL)
        self.directives = [m.content for m in procedural]
        print(f"Reflection engine loaded {len(self.directives)} existing directives.")

    def reflect(self, user_message, assistant_response, interaction_id):
        """Run reflection pass at all depths. Returns list of new memories created."""
        new_memories = []

        # Depth 0 — Factual: What happened?
        d0 = self._depth_0(user_message, assistant_response)
        if d0:
            new_memories.append(d0)

        # Depth 1 — Analytical: What patterns emerge?
        d1 = self._depth_1(user_message, assistant_response)
        if d1:
            new_memories.append(d1)

        # Depth 2 — Prescriptive: What should change?
        d2 = self._depth_2(d0, d1)
        if d2:
            new_memories.append(d2)

        # Depth 3 — Meta-coherence: Are directives consistent?
        if d2:
            d3 = self._depth_3()
            if d3:
                new_memories.append(d3)

        # Store all new memories
        for mem in new_memories:
            mem.source_interaction_id = interaction_id
            self.memory.store(mem)

        return new_memories

    def _depth_0(self, user_message, assistant_response):
        """Factual: evaluate what happened in this interaction."""
        prompt = (
            "[INST] You are a self-reflection module analyzing an interaction.\n\n"
            f"Interaction:\nUser: {user_message}\nAssistant: {assistant_response}\n\n"
            "Analyze this interaction factually:\n"
            "1. Was the response accurate and relevant?\n"
            "2. Did it address what the user actually asked?\n"
            "3. Were there any errors or misunderstandings?\n\n"
            "Be brief and specific. One paragraph. [/INST]"
        )

        analysis = self.engine.generate_structured(prompt, max_new_tokens=200)

        return MemoryUnit(
            content=f"Reflection D0: {analysis}",
            memory_type=MemoryType.REFLECTIVE,
            salience_score=0.4,
            confidence=0.6,
            tags=["reflection", "depth_0", "factual"],
        )

    def _depth_1(self, user_message, assistant_response):
        """Analytical: identify patterns across recent interactions."""
        recent = self.memory.retrieve(user_message, top_k=5)
        if len(recent) < 2:
            return None

        memory_context = "\n".join(
            [f"- [{m.memory_type.value}] {m.content[:150]}" for m in recent]
        )

        prompt = (
            "[INST] You are a self-reflection module analyzing patterns.\n\n"
            f"Current interaction:\nUser: {user_message}\nAssistant: {assistant_response}\n\n"
            f"Recent memories:\n{memory_context}\n\n"
            "What patterns do you notice?\n"
            "- Recurring user needs or preferences\n"
            "- Consistent strengths or weaknesses in responses\n"
            "- Emerging themes across interactions\n\n"
            "Be brief and specific. One paragraph. [/INST]"
        )

        analysis = self.engine.generate_structured(prompt, max_new_tokens=200)

        return MemoryUnit(
            content=f"Reflection D1: {analysis}",
            memory_type=MemoryType.REFLECTIVE,
            salience_score=0.5,
            confidence=0.5,
            tags=["reflection", "depth_1", "analytical"],
        )

    def _depth_2(self, d0_memory, d1_memory):
        """Prescriptive: generate behavioral directives from analysis."""
        if not d0_memory and not d1_memory:
            return None

        context_parts = []
        if d0_memory:
            context_parts.append(f"Factual analysis: {d0_memory.content}")
        if d1_memory:
            context_parts.append(f"Pattern analysis: {d1_memory.content}")

        current_directives = "\n".join(
            [f"- {d}" for d in self.directives]
        ) if self.directives else "None yet."

        prompt = (
            "[INST] You are a self-reflection module generating behavioral directives.\n\n"
            f"Analysis:\n" + "\n".join(context_parts) + "\n\n"
            f"Current directives:\n{current_directives}\n\n"
            "Based on the analysis, should any NEW directive be added? A directive is a specific behavioral rule like:\n"
            '- "Keep responses under 3 sentences unless asked for detail"\n'
            '- "When user mentions Contact Front, ask about progress"\n\n'
            "Rules:\n"
            "- Only propose a directive if the analysis clearly supports it\n"
            "- Do not duplicate existing directives\n"
            "- If no new directive is needed, respond with exactly: NO_NEW_DIRECTIVE\n\n"
            "If proposing a directive, state it as a single clear sentence. [/INST]"
        )

        result = self.engine.generate_structured(prompt, max_new_tokens=100)

        if "NO_NEW_DIRECTIVE" in result.upper():
            return None

        # Clean up the directive
        directive = result.strip().split("\n")[0].strip()
        if len(directive) < 10 or len(directive) > 200:
            return None

        self.directives.append(directive)

        return MemoryUnit(
            content=directive,
            memory_type=MemoryType.PROCEDURAL,
            salience_score=0.8,
            confidence=0.7,
            tags=["reflection", "depth_2", "directive", "auto_generated"],
        )

    def _depth_3(self):
        """Meta-coherence: check directives for contradictions."""
        if len(self.directives) < 2:
            return None

        directive_list = "\n".join(
            [f"{i+1}. {d}" for i, d in enumerate(self.directives)]
        )

        prompt = (
            "[INST] You are a coherence checker for behavioral directives.\n\n"
            f"Current directives:\n{directive_list}\n\n"
            "Check for:\n"
            "1. Contradictions between directives\n"
            "2. Redundancies (two directives saying the same thing)\n"
            "3. Directives that are too vague to be actionable\n\n"
            "If all directives are coherent, respond with exactly: COHERENT\n\n"
            "If there are issues, briefly describe each one and which directive numbers are involved. [/INST]"
        )

        result = self.engine.generate_structured(prompt, max_new_tokens=200)

        if "COHERENT" in result.upper():
            return None

        return MemoryUnit(
            content=f"Reflection D3 — Coherence issue: {result}",
            memory_type=MemoryType.REFLECTIVE,
            salience_score=0.6,
            confidence=0.5,
            tags=["reflection", "depth_3", "coherence"],
        )

    def get_directives(self):
        """Return current active directives for context assembly."""
        return self.directives.copy()

In [ ]:
!ls /workspace/Projects/cultivated-learning/core/interaction_loop_backup.py

In [ ]:
%%writefile /workspace/Projects/cultivated-learning/core/interaction_loop.py
import time
import json
import uuid
from core.memory_store import MemoryUnit, MemoryType
from core.reflection import ReflectionEngine


class InteractionLoop:
    """Main loop: assembles context, generates response, stores memories, reflects."""

    def __init__(self, engine, memory_store, assembler, log_dir=None, reflect=True):
        self.engine = engine
        self.memory = memory_store
        self.assembler = assembler
        self.log_dir = log_dir
        self.history = []
        self.interaction_count = 0
        self.reflect_enabled = reflect
        self.reflection_engine = None

        if self.reflect_enabled:
            self.reflection_engine = ReflectionEngine(engine, memory_store)

    def chat(self, user_message):
        self.interaction_count += 1
        interaction_id = str(uuid.uuid4())
        start_time = time.time()

        # Get directives from reflection engine if available
        directives = None
        if self.reflection_engine:
            directives = self.reflection_engine.get_directives()

        # Assemble context
        prompt = self.assembler.assemble(
            user_message=user_message,
            conversation_history=self.history,
            directives=directives,
        )

        # Generate response
        response = self.engine.generate(prompt)

        elapsed = time.time() - start_time

        # Update conversation history
        self.history.append({"role": "user", "content": user_message})
        self.history.append({"role": "assistant", "content": response})

        # Keep history manageable (last 10 turns = 20 messages)
        if len(self.history) > 20:
            self.history = self.history[-20:]

        # Store episodic memory
        episodic = MemoryUnit(
            content=f"User: {user_message}\nAssistant: {response[:200]}",
            memory_type=MemoryType.EPISODIC,
            source_interaction_id=interaction_id,
            salience_score=0.5,
            tags=["interaction"],
        )
        self.memory.store(episodic)

        # Reflection pass (async in future, synchronous for now)
        if self.reflection_engine:
            try:
                reflections = self.reflection_engine.reflect(
                    user_message, response, interaction_id
                )
                if reflections:
                    print(f"  Reflection: {len(reflections)} new memories generated")
            except Exception as e:
                print(f"  Reflection error (non-fatal): {e}")

        # Log interaction
        if self.log_dir:
            self._log(interaction_id, user_message, response, prompt, elapsed)

        return response

    def feedback(self, rating, correction=None):
        """Process explicit feedback on the last interaction."""
        if len(self.history) < 2:
            print("No interaction to rate.")
            return

        last_user = self.history[-2]["content"]
        last_assistant = self.history[-1]["content"]

        # Adjust salience of recent episodic memories
        recent = self.memory.retrieve(last_user, top_k=3)
        delta = (rating - 3) * 0.1
        for mem in recent:
            self.memory.adjust_salience(mem.id, delta)

        # Store correction as high-salience semantic memory
        if correction:
            correction_mem = MemoryUnit(
                content=f"CORRECTION: {correction}",
                memory_type=MemoryType.SEMANTIC,
                salience_score=0.9,
                confidence=1.0,
                tags=["user_correction", "high_priority"],
            )
            self.memory.store(correction_mem)
            print(f"Stored correction: {correction}")

        print(f"Feedback recorded: rating={rating}, adjusted {len(recent)} memories by {delta:+.2f}")

    def status(self):
        stats = self.memory.get_stats()
        directive_count = len(self.reflection_engine.get_directives()) if self.reflection_engine else 0
        return {
            "interaction_count": self.interaction_count,
            "history_length": len(self.history),
            "memory": stats,
            "active_directives": directive_count,
            "reflection_enabled": self.reflect_enabled,
        }

    def _log(self, interaction_id, user_message, response, prompt, elapsed):
        import os
        os.makedirs(self.log_dir, exist_ok=True)
        log_entry = {
            "id": interaction_id,
            "timestamp": time.time(),
            "user_message": user_message,
            "response": response,
            "prompt_tokens": self.engine.count_tokens(prompt),
            "response_tokens": self.engine.count_tokens(response),
            "elapsed_seconds": round(elapsed, 2),
            "memory_count": self.memory.collection.count(),
        }
        path = os.path.join(self.log_dir, f"{interaction_id}.json")
        with open(path, "w") as f:
            json.dump(log_entry, f, indent=2)

In [ ]:
from engine.inference import InferenceEngine

engine = InferenceEngine("/workspace/models/results/Mistral-7B-Instruct-v0.3")
engine.load()

In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")

import importlib
import core.reflection
import core.interaction_loop
import core.memory_store
import core.context_assembler
importlib.reload(core.reflection)
importlib.reload(core.interaction_loop)
importlib.reload(core.memory_store)
importlib.reload(core.context_assembler)

from core.memory_store import MemoryStore
from core.context_assembler import ContextAssembler
from core.interaction_loop import InteractionLoop

memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

assembler = ContextAssembler(engine=engine, memory_store=memory)

loop = InteractionLoop(
    engine=engine,
    memory_store=memory,
    assembler=assembler,
    log_dir="/workspace/Projects/cultivated-learning/data/interaction_log"
)

print("\nTesting reflection...")
response = loop.chat("What do you remember about me?")
print(f"\nResponse: {response}")

In [ ]:
from core.memory_store import MemoryType

reflective = memory.retrieve_by_type(MemoryType.REFLECTIVE)
for m in reflective:
    print(f"[{m.salience_score:.2f}] {m.content[:120]}...")
    print(f"  Tags: {m.tags}\n")

print("Active directives:")
for d in loop.reflection_engine.get_directives():
    print(f"  - {d}")